# Test Suite — Copy-on-Write vs Merge-on-Read

Ejecuta los 13 escenarios dos veces: una en **copy-on-write** y otra en **merge-on-read**.
Al final compara resultados: conflictos, tipos de error y tiempos.

**Run All** y espera — tarda ~15-20 min en total.

In [ ]:
import subprocess, sys, os, threading, json, glob, time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

CATALOG         = "players"
DATABASE        = "concurrency_tests"
TARGET          = f"{CATALOG}.{DATABASE}.concurrent_table"
CONFLICT_ROW_ID = 7

SCENARIOS = [
    ("s01_ins_ins",  "insert  vs insert",  ("insert", "0"),                      ("insert", "0"),                      "✅✅ sin conflicto"),
    ("s02_ins_read", "insert  vs read",    ("insert", "0"),                      ("read",   "0"),                      "✅✅ sin conflicto"),
    ("s03_ins_upd",  "insert  vs update",  ("insert", "0"),                      ("update", str(CONFLICT_ROW_ID)),     "✅✅ sin conflicto"),
    ("s04_ins_del",  "insert  vs delete",  ("insert", "0"),                      ("delete", str(CONFLICT_ROW_ID)),     "✅✅ sin conflicto"),
    ("s05_ins_mrg",  "insert  vs merge",   ("insert", "0"),                      ("merge",  str(CONFLICT_ROW_ID)),     "✅✅ sin conflicto"),
    ("s06_read_read","read    vs read",    ("read",   "0"),                      ("read",   "0"),                      "✅✅ sin conflicto"),
    ("s07_read_upd", "read    vs update",  ("read",   "0"),                      ("update", str(CONFLICT_ROW_ID)),     "✅✅ sin conflicto"),
    ("s08_upd_upd",  "update  vs update",  ("update", str(CONFLICT_ROW_ID)),     ("update", str(CONFLICT_ROW_ID)),     "⚡ conflicto"),
    ("s09_upd_del",  "update  vs delete",  ("update", str(CONFLICT_ROW_ID)),     ("delete", str(CONFLICT_ROW_ID)),     "⚡ conflicto"),
    ("s10_upd_mrg",  "update  vs merge",   ("update", str(CONFLICT_ROW_ID)),     ("merge",  str(CONFLICT_ROW_ID)),     "⚡ conflicto"),
    ("s11_del_del",  "delete  vs delete",  ("delete", str(CONFLICT_ROW_ID)),     ("delete", str(CONFLICT_ROW_ID)),     "⚡ conflicto"),
    ("s12_del_mrg",  "delete  vs merge",   ("delete", str(CONFLICT_ROW_ID)),     ("merge",  str(CONFLICT_ROW_ID)),     "⚡ conflicto"),
    ("s13_mrg_mrg",  "merge   vs merge",   ("merge",  str(CONFLICT_ROW_ID)),     ("merge",  str(CONFLICT_ROW_ID)),     "⚡ conflicto"),
]

WRITE_MODES = [
    ("copy-on-write", "COW"),
    ("merge-on-read", "MOR"),
]

print(f"Escenarios: {len(SCENARIOS)}  x  Modos: {len(WRITE_MODES)}  =  {len(SCENARIOS)*len(WRITE_MODES)} runs")

In [ ]:
# ── SPARK NOTEBOOK ─────────────────────────────────────────
def get_spark_notebook():
    builder = SparkSession.builder.appName("test_suite_setup")
    builder = builder.config("spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.367")
    builder = builder.config("spark.jars.excludes",
        "org.apache.hadoop:hadoop-client-runtime,org.apache.hadoop:hadoop-client-api")
    builder = builder.config("spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    for k, v in [
        ("spark.sql.catalog.players",                          "org.apache.iceberg.spark.SparkCatalog"),
        ("spark.sql.catalog.players.type",                     "rest"),
        ("spark.sql.catalog.players.uri",                      "http://172.16.58.11:32688"),
        ("spark.sql.catalog.players.warehouse",                "s3://warehouse/"),
        ("spark.sql.catalog.players.io-impl",                  "org.apache.iceberg.aws.s3.S3FileIO"),
        ("spark.sql.catalog.players.s3.endpoint",              "http://172.16.58.11:31224"),
        ("spark.sql.catalog.players.s3.region",                "us-east-1"),
        ("spark.sql.catalog.players.s3.path-style-access",     "true"),
        ("spark.sql.catalog.players.client.region",            "us-east-1"),
        ("spark.hadoop.fs.s3a.endpoint",                       "http://172.16.58.11:31224"),
        ("spark.hadoop.fs.s3a.endpoint.region",                "us-east-1"),
        ("spark.hadoop.fs.s3a.access.key",                     "GK5f421d5f440758f74b0e0312"),
        ("spark.hadoop.fs.s3a.secret.key",                     "409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d"),
        ("spark.hadoop.fs.s3a.path.style.access",              "true"),
        ("spark.hadoop.fs.s3a.impl",                           "org.apache.hadoop.fs.s3a.S3AFileSystem"),
        ("spark.sql.catalog.players.s3.checksum-algorithm",    "NONE"),
        ("spark.sql.catalog.players.s3.streaming-upload-enabled", "false"),
        ("spark.sql.catalog.players.s3.chunked-encoding-enabled", "false"),
        ("spark.sql.catalog.players.s3.payload-signing-enabled",  "true"),
        ("spark.sql.catalog.players.s3.http-client-type",      "urlconnection"),
        ("spark.executor.extraJavaOptions",                    "-Daws.requestChecksumCalculation=when_required"),
        ("spark.driver.extraJavaOptions",                      "-Daws.requestChecksumCalculation=when_required"),
        ("spark.sql.catalog.players.s3.access-key-id",        "GK5f421d5f440758f74b0e0312"),
        ("spark.sql.catalog.players.s3.secret-access-key",    "409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d"),
    ]:
        builder = builder.config(k, v)
    return builder.getOrCreate()

spark = get_spark_notebook()
spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("id",     IntegerType()),
    StructField("source", StringType()),
    StructField("value",  StringType()),
    StructField("ts",     TimestampType()),
])
print("✅ SparkSession lista")

In [ ]:
# ── HELPERS ─────────────────────────────────────────────────

clean_env = {k: v for k, v in os.environ.items()
             if "SPARK" not in k and "PYSPARK" not in k}

def setup_table(write_mode):
    """Crea la tabla desde cero con el modo indicado."""
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{DATABASE}")
    spark.sql(f"DROP TABLE IF EXISTS {TARGET} PURGE")
    spark.sql(f"""
        CREATE TABLE {TARGET} (
            id INT, source STRING, value STRING, ts TIMESTAMP
        ) USING iceberg
        TBLPROPERTIES (
            'write.update.mode'        = '{write_mode}',
            'write.merge.mode'         = '{write_mode}',
            'write.delete.mode'        = '{write_mode}',
            'commit.retry.num-retries' = '4',
            'commit.retry.min-wait-ms' = '100',
            'commit.retry.max-wait-ms' = '2000'
        )
    """)
    seed = [(i, "seed", f"val_{i}", datetime.now()) for i in range(1, 21)]
    spark.createDataFrame(seed, schema).writeTo(TARGET).append()

def reset_conflict_row():
    """Restaura la fila CONFLICT_ROW_ID si fue borrada por el test anterior."""
    spark.catalog.refreshTable(TARGET)
    if spark.table(TARGET).filter(f"id = {CONFLICT_ROW_ID}").count() == 0:
        spark.createDataFrame(
            [(CONFLICT_ROW_ID, "seed", f"val_{CONFLICT_ROW_ID}", datetime.now())], schema
        ).writeTo(TARGET).append()

def run_scenario(scenario_id, w0, w1, mode_tag):
    """Lanza 2 workers con barrera y recoge resultados."""
    # id único por modo para que las barreras no colisionen
    sid = f"{mode_tag}_{scenario_id}"
    barrier_go = f"/tmp/{sid}_go"

    for f in glob.glob(f"/tmp/{sid}_*"):
        try: os.remove(f)
        except: pass

    results = []
    lock = threading.Lock()

    def stream(proc, wid):
        for line in proc.stdout:
            line = line.strip()
            if not line: continue
            try:
                r = json.loads(line)
                r["write_mode"] = mode_tag
                with lock: results.append(r)
            except json.JSONDecodeError:
                if any(k in line for k in ["ERROR", "Exception", "INFO:"]):
                    print(f"    [W{wid}] {line}", flush=True)

    processes, threads = [], []
    for i, (op, target) in enumerate([w0, w1]):
        proc = subprocess.Popen(
            [sys.executable, "worker.py", str(i), op, target, sid],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=clean_env
        )
        t = threading.Thread(target=stream, args=(proc, i), daemon=True)
        t.start()
        processes.append(proc); threads.append(t)

    t0 = time.time()
    while True:
        if len(glob.glob(f"/tmp/{sid}_worker_*_ready")) >= 2: break
        if time.time() - t0 > 300: raise TimeoutError(f"Timeout {sid}")
        time.sleep(0.5)

    time.sleep(0.1)
    open(barrier_go, "w").close()

    for proc in processes: proc.wait()
    for t in threads: t.join(timeout=5)

    for f in glob.glob(f"/tmp/{sid}_*"):
        try: os.remove(f)
        except: pass

    return results

print("✅ Helpers listos")

In [ ]:
# ── EJECUTAR TODOS LOS MODOS Y ESCENARIOS ──────────────────

all_results = []   # todos los registros individuales
all_summary = []   # una fila por (modo, escenario)

NEEDS_RESET = {"update", "delete", "merge"}

for write_mode, mode_tag in WRITE_MODES:
    print(f"\n{'═'*65}")
    print(f"  MODO: {write_mode} ({mode_tag})")
    print(f"{'═'*65}")

    setup_table(write_mode)
    print(f"  Tabla creada en modo {write_mode}")

    for scenario_id, desc, w0, w1, expected in SCENARIOS:
        # Restaura fila de conflicto si alguna operación puede haberla tocado
        if w0[0] in NEEDS_RESET or w1[0] in NEEDS_RESET:
            reset_conflict_row()

        t_start = time.time()
        results = run_scenario(scenario_id, w0, w1, mode_tag)
        elapsed = time.time() - t_start

        all_results.extend(results)

        if not results:
            print(f"  ⚠️  {scenario_id}: sin resultados")
            continue

        df_sc   = pd.DataFrame(results)
        ok_n    = (df_sc["status"] == "ok").sum()
        conf_n  = df_sc["status"].str.contains("conflict").sum()
        err_n   = (df_sc["status"] == "error").sum()
        avg_ms  = df_sc["duration_ms"].mean()
        tipos   = df_sc[df_sc["status"].str.contains("conflict")]["status"].value_counts().to_dict() if conf_n > 0 else {}

        icon = "✅" if conf_n == 0 else "⚡"
        tipo_str = f" [{', '.join(f'{k}:{v}' for k,v in tipos.items())}]" if tipos else ""
        print(f"  {icon} {scenario_id:<15} ok={ok_n:>2}  conf={conf_n:>2}  "
              f"avg={avg_ms:>7.1f}ms  {elapsed:.1f}s{tipo_str}")

        all_summary.append({
            "mode"      : mode_tag,
            "scenario"  : scenario_id,
            "desc"      : desc,
            "expected"  : expected,
            "ok"        : int(ok_n),
            "conflicts" : int(conf_n),
            "errors"    : int(err_n),
            "avg_ms"    : round(avg_ms, 1),
            "elapsed_s" : round(elapsed, 1),
            "conflict_types": list(tipos.keys()),
        })

print(f"\n{'═'*65}")
print("  TODOS LOS RUNS COMPLETADOS")
print(f"  Total registros: {len(all_results)}")
print(f"{'═'*65}")

In [ ]:
# ── COMPARATIVA FINAL ──────────────────────────────────────

df_sum = pd.DataFrame(all_summary)
df_all = pd.DataFrame(all_results)

# ── Tabla pivot: conflictos COW vs MOR ─────────────────────
pivot_conf = df_sum.pivot_table(
    index=["scenario", "desc", "expected"],
    columns="mode",
    values="conflicts",
    aggfunc="sum"
).fillna(0).astype(int).reset_index()

print("=" * 75)
print("CONFLICTOS: COW vs MOR (por escenario)")
print("=" * 75)
print(pivot_conf.to_string(index=False))

# ── Tabla pivot: tiempos medios COW vs MOR ─────────────────
pivot_ms = df_sum.pivot_table(
    index=["scenario", "desc"],
    columns="mode",
    values="avg_ms",
    aggfunc="mean"
).round(1).reset_index()

# Diferencia relativa entre modos
if "COW" in pivot_ms.columns and "MOR" in pivot_ms.columns:
    pivot_ms["diff_ms"]   = (pivot_ms["MOR"] - pivot_ms["COW"]).round(1)
    pivot_ms["MOR_vs_COW"] = (pivot_ms["diff_ms"] / pivot_ms["COW"] * 100).round(1).astype(str) + "%"

print("\n" + "=" * 75)
print("TIEMPOS MEDIOS (ms): COW vs MOR")
print("=" * 75)
print(pivot_ms.to_string(index=False))

# ── Tipos de conflicto por modo ────────────────────────────
print("\n" + "=" * 75)
print("TIPOS DE CONFLICTO POR MODO")
print("=" * 75)
print("  copy-on-write  → CommitFailedException  (el fichero reescrito ya cambió)")
print("  merge-on-read  → ValidationException    (encontró delete files conflictivos)")
print()
conflict_rows = df_all[df_all["status"].str.contains("conflict", na=False)]
if not conflict_rows.empty:
    print(conflict_rows.groupby(["write_mode", "operation", "status"])
          .size().rename("count").to_string())

# ── Veredicto ──────────────────────────────────────────────
print("\n" + "=" * 75)
print("VEREDICTO")
print("=" * 75)
for _, row in pivot_conf.iterrows():
    cow_c = row.get("COW", 0)
    mor_c = row.get("MOR", 0)
    esperaba = "conflicto" in row["expected"]

    cow_ok = (cow_c > 0) == esperaba
    mor_ok = (mor_c > 0) == esperaba

    cow_icon = "✅" if cow_ok else "❌"
    mor_icon = "✅" if mor_ok else "❌"

    diff = ""
    if cow_c > 0 and mor_c == 0:
        diff = "  ← solo COW conflicta"
    elif mor_c > 0 and cow_c == 0:
        diff = "  ← solo MOR conflicta"
    elif cow_c > 0 and mor_c > 0:
        diff = f"  ← ambos conflictan (COW={cow_c} MOR={mor_c})"

    print(f"  COW{cow_icon} MOR{mor_icon}  {row['scenario']:<15} {row['desc']}{diff}")